## Differences between Docker and Virtual Machines (VMs)

![Docker vs VMs in ML](image.png)

#### 1. Startup & Speed

* **VMs:** Boot an entire OS $\rightarrow$ slow startup.
* **Docker:** Shares the host kernel $\rightarrow$ containers start in seconds.
    * 👉 Faster iteration when testing ML models.

#### 2. Resource Efficiency

* **VMs:** Each VM runs its own OS $\rightarrow$ heavy RAM/CPU usage.
* **Docker:** Lightweight, only isolates processes $\rightarrow$ more containers per machine.
    * 👉 You can run multiple ML experiments side-by-side without exhausting resources.

#### 3. Portability & Reproducibility

* **VMs:** Images are large, harder to move.
* **Docker:** Small, layered images $\rightarrow$ easy to share and deploy.
    * 👉 Your ML environment (Python, CUDA, PyTorch) runs identically on laptop, server, or cloud.

#### 4. Isolation & Workflow

* **VMs:** Stronger isolation but clunky dev workflow.
* **Docker:** High isolation with smoother workflow (easy rebuilds, versioning).
    * 👉 Perfect for ML pipelines where you need reproducibility but also agility.

#### 5. GPU & ML Integration

* Docker integrates with **NVIDIA Container Toolkit** for GPU passthrough.
* You can run CUDA, TensorFlow, PyTorch inside containers without VM overhead.
    * 👉 Essential for ML workloads that demand GPU acceleration.


## Advantages of using Docker in AI & ML workflows

#### 1. Consistent & Reproducible Environments

* Docker ensures the same environment everywhere (local machine, server, cloud).
* No "works on my machine" problem – ML code runs identically across setups.

#### 2. Simplified Dependency Management

* ML projects often need specific Python, CUDA, or library versions.
* Docker makes it easy to package these dependencies into one container.

#### 3. Seamless Experimentation & Model Versioning

* You can spin up containers for different experiments quickly.
* Helps track versions of models and environments without conflicts.

##### Q. We can performs model versioning by use of MlFlow & LeakeFS. So why we required Docker for model versioning?

While tools like `MLflow` version the model weights (the `artifacts`) and `lakeFS` versions the raw data, Docker contributes to model versioning by `versioning the runtime environment`.
Without Docker, you can version a model perfectly, but if the underlying OS, C++ libraries, or Python packages shift in production, that versioned model will output different predictions or fail entirely.

**The Problem**: If `MLflow` tells you a model is Version 3, but that version relies on `scikit-learn 1.2` and your production server upgrades to `scikit-learn 1.5`, the model might break or give silent, incorrect predictions due to underlying algorithmic changes. `Docker` packages the model `artifact`, the exact Python version, the precise library dependencies, and the OS drivers (like specific CUDA versions for GPUs) into an immutable Docker Image. Once built, this image cannot change.

#### 4. Portable Deployment (Cloud, Edge, Anywhere)

* A Dockerized ML app can be deployed on AWS, GCP, Azure, or even edge devices.
* Portability is crucial when moving from research $\rightarrow$ production.

#### 5. Modular Microservices for Scalable Inference

* ML inference can be split into microservices (e.g., preprocessing, model serving, postprocessing).
* Docker makes scaling these services easier with orchestration tools like Kubernetes.


## Stages in the ML Workflow & Docker’s Role

![Docker with ML workflow](image-1.png)

![Common Tools + Docker in ML](image-3.png)

Docker provides the containerized environment (dependencies, CUDA, Python, libraries), while MLflow handles experiment tracking, model packaging, and deployment. 

Docker + ML Together they give 

- **Reproducibility** → Docker ensures the same environment everywhere.
- **Experiment Tracking** → MLflow logs runs, metrics, and artifacts.
- **Model Packaging** → MLflow can package models into Docker images.
- **Deployment** → Dockerized MLflow models can be deployed as REST APIs or scaled with Kubernetes.

## Docker = environment consistency and MLflow = experiment + model lifecycle management.

#### 1. Data Collection & Exploration

* **Reproducible Preprocessing:** Docker ensures you have a reproducible environment for data preprocessing (same Python, Pandas, NumPy versions).
* **Easier Collaboration:** Teammates can run the same container and get identical results.

#### Q. Docker used in Data Collection & Exploration. How ever I use LakeFS for data versioning and control. So how Docker is differ from LakeFS?

Docker and LakeFS both deal with reproducibility and version control in ML workflows, but they operate at different layers of the stack. Docker and LakeFS Complement Each Other.

**Docker**: Ensures your training environment is identical everywhere. Example: same Python, CUDA, PyTorch versions across dev, test, and prod. You can run `docker run --gpus all my-ml-image` and get consistent behavior.

**LakeFS**: Ensures your data is versioned and reproducible. Example: you can branch your dataset (lakefs branch experiment-1) and commit changes. Enables rollback to previous data states, just like Git commits.
* Docker = reproducible code + environment
* LakeFS = reproducible data + metadata

![Workflow Integration Docker + LakeFS](image-2.png)

#### 2. Model Development & Training

* **Dependency Bundling:** Package CUDA, PyTorch/TensorFlow, and dependencies inside a container.
* **Hardware Agnostic:** Guarantees consistency across local machine, cloud GPU, or HPC cluster.
* **No Version Conflicts:** Avoids "dependency hell" when training models.

#### 3. Experiment Tracking

* **Isolated Logging:** Run MLflow, Weights & Biases, or custom logging inside Docker containers.
* **Reproducible Runs:** Each experiment can be isolated in its own container, making results reproducible.

#### 4. Model Packaging

* **Inference Bundling:** Bundle trained models with inference code into a Docker image.
* **Runtime Consistency:** Keeps the runtime environment consistent (same Python, same libraries).
* **Simplified Versioning:** Makes versioning easier—each image tag corresponds to a model version.

#### 5. Model Deployment

* **Microservice Ready:** Deploy the Dockerized model as a microservice (e.g., FastAPI, Flask).
* **Orchestration Friendly:** Works seamlessly with Kubernetes for scaling inference.
* **Highly Portable:** Portable — deploy on cloud, edge devices, or on-prem servers.

#### 6. Monitoring & Updates

* **Rolling Updates:** Containers simplify rolling updates (replace old image with new one).
* **Sidecar Monitoring:** Monitoring tools (Prometheus, Grafana) can run in Docker alongside ML services.
* **Safe Failures:** Easy rollback if a new model version misbehaves.



![Popular Use Cases for Docker in ML](image-4.png)

![Who's Using Docker for AI/ML?](image-5.png)

## Docker in the world of LLMs / Agentic AI

Running Models with Docker Model Runner means you can take any trained ML model, wrap it in a Docker container, and instantly expose it as a REST API — without writing serving code yourself.

- Dockerized Abstraction of Any Trained Model:
    - You can package a trained ML model (PyTorch, TensorFlow, XGBoost, etc.) into a Docker container.
      This container includes all dependencies, so you don’t worry about environment mismatches.
      ```cmd
       This spins up a container exposing the model on port 8080.
      docker run -p 8080:8080 ghcr.io/mlc-ai/model-runner:latest
      ```
- Launch a Ready-to-Serve REST API in Seconds:      
    - No need to write custom `Flask/FastAPI` code.The container automatically provides a REST API endpoint for inference. You can send HTTP requests (POST /predict) to get predictions from the model.
- Framework Agnostic:
  - Works with all major ML frameworks (TensorFlow, PyTorch, XGBoost, Scikit-learn). You don’t need to change your model code — just export the trained model and run it with the Docker Model Runner.

Before Docker: You’d have to write custom serving code, set up dependencies, and manage deployment.But With `Docker Model Runner, Package → Run → Serve → Scale`. One command gives you a production-ready API.
Portable across cloud, edge, or local machines.

## Docker + MCP Tooling:

As we know Model Context Protocol (`MCP`) is a framework that allows AI models to connect with external tools (like Kubernetes, Terraform, CLI agents). It acts like a “hub” that standardizes communication between models and systems.

By using the `Docker + MCP Integration` you can able to MCP servers inside Docker containers, you get a self‑hosted toolkit that’s portable and reproducible.

```cmd
docker run -p 3000:3000 realops/kubernetes-mcp-server:latest
This spins up a Kubernetes MCP server inside Docker, exposing it on port 3000.
```

![Docker + MCP Tooling](image-6.png)

![Docker + MCP digital architecture diagram](image-7.png)

## NVIDIA NIM

NVIDIA NIM is a suite of pre‑built, GPU‑accelerated inference microservices that make it easy to deploy large language models (LLMs) and other AI models anywhere — on cloud, data centers, workstations, or edge devices — with optimized performance and industry‑standard APIs.

Pre‑built Microservices: Containers with foundation models (LLMs, vision, speech, simulation). Includes optimized inference engines: TensorRT, TensorRT‑LLM, vLLM, SGLang. Ready‑to‑deploy with minimal setup.

REST and gRPC endpoints for inference. Easy integration into apps, agents, and workflows. Models can be deployed in under 5 minutes.

Supports self‑hosting (via Docker/Kubernetes) or managed endpoints (e.g., Hugging Face).

Optimized for low latency and high throughput.

## NVIDIA NIM can be deployed using Docker 

ToDo

## Agentic AI + Docker

TODO

## Docker Life cycle

![Docker Life cycle](image-9.png)

- **Pull Image from Registry**: When you run docker run ..., Docker first checks if the image exists locally.If not, it pulls the image from a registry (e.g., Docker Hub, GitHub Container Registry).
- **Create Container**: Docker uses the image to create a container object. This container is like a lightweight, isolated runtime environment.
- Attach Container:
	- If you use -it, Docker attaches your terminal to the container’s input/output. This lets you interact with the container directly.
- Set Up Network & Version:
	- Docker assigns the container to a network (bridge by default). It configures environment variables, mounts volumes, and sets runtime options.
- Start Container → Running State:
	- The container process starts (e.g., MLflow server, Nginx, Python script). At this point, the container is “alive” and executing.

## Docker Commands

1. docker version
    - docker --version
	- list out docker image `docket image ls`
	- Docker Engine status `docker info`
	- Run a test container `docker run hello-world`

	![about docker command](image-8.png)
Note: you can also replace Namespace by <yuor library> i.e. above command can be written as
```
docker.io/library/mlflow:latest
```

2. Launch mlflow using a container:
	- Get mlflow from Docker Hub
		- `docker pull mlflow/mlflow:latest`
	- GitHub Container Registry (GHCR) [ghcr.io docker container registry]
		- docker pull ghcr.io/mlflow/mlflow:latest
	- You can also pull specific versions
		- docker pull ghcr.io/mlflow/mlflow:v3.16.0-full	
	- analyzes container images for vulnerabilities, outdated packages, and best practices:
		- docker scout quickview ghcr.io/mlflow/mlflow:latest
			- `docker scout` → This is Docker’s security and insights tool. It analyzes container images for vulnerabilities, outdated packages, and best practices.
			- `quickview` → A subcommand that gives you a summary report of the image. It’s like a “health check” — showing 
			CVEs (Common Vulnerabilities and Exposures), base image details, and recommendations.
			- `ghcr.io/mlflow/mlflow:latest` → This specifies the image you want to analyze
	- run mlflow on your local host
		- Befor run check mlflow versions on docker 
			* `docker run --rm ghcr.io/mlflow/mlflow:latest mlflow --version`
		- Basic command Run MLflow server
			* docker run <...> mlflow server
			* `docker run -it --rm -p 5000:5555 ghcr.io/mlflow/mlflow:v3.16.0 mlflow server`
				* `-it` <interactive: Keeps STDIN open> <tty: Allocates a pseudo‑terminal (like a shell).>
				* `--rm` (remove) Automatically deletes the container when it exits.
				* `part -p 5000:5555` is the port mapping option. -p <host_port>:<container_port>. Host port (5000) → The port on your local machine (Windows/Mac/Linux). Container port (5555) → The port inside the container where MLflow server is listening.
				* You can now open your browser at: `http://<localhost ip>:5000`. See on which local ip port running your docker container
		- kill running server
			* Find container id: `docker ps`
			* kill running server by CONTAINER ID `docker stop 474195838f44`
	- launch the container in detached mode:
		* `docker run -d -it --rm -p 5555:5000 --name mlflowX ghcr.io/mlflow/mlflow:v3.16.0 mlflow server --host 0.0.0.0`
			* -d `-d` : run container in detached mode
			* --name mlflowX : sets the name of the container as `mlflow` instead of a auto generated random name
			* --host 0.0.0.0 : tells MLflow to listen on all interfaces inside the container.
			* on google crome run mlflow with `http://127.0.0.1:5555` or `http://localhost:5555`		
	- launch the container in detached mode with namespace [(**mlflowX**)]:
		* `docker run -d -it --rm -p 5555:5000 --name mlflowX ghcr.io/mlflow/mlflow:v3.16.0 mlflow server --host 0.0.0.0`
			* docker ps
			* docker logs mlflowX
			* docker stop mlflowX
	- check the logs for mlflow container [name space set **mlflowX**. If not set then use default i.e. **mlflow**]
		* Use `docker logs mlflowX` if you just want a quick snapshot (e.g., check if MLflow started successfully).
		* Use `docker logs -f mlflowX` if you want to monitor the container continuously (e.g., see incoming API calls or debug startup issues).
		* `Ctrl+C` exits the log stream
	- **open a shell inside a running Docker container**:
		* `docker exec -it mlflowX sh` where mlflowX is  container name you gave. If user not given this, it could be default name provided by docker.
			* `docker exec` Tells Docker to run a command inside an already running container. Unlike docker run, it doesn’t start a new container
			* `sh` starts a shell (like a lightweight terminal)
			* From there, you’ll be inside the container’s filesystem and environment, able to run commands like `ls`, `cat`, `ps`, `python`. It just like linux terminal command. 
			* note: ***Inside the container shell: You can only run commands available in that container’s OS (like ls, cat, ps, python, etc.). But not Docker commands — those belong to the host.***
3. Run Jupyter for ML Notebooks with a local Volume
	* `docker run -d -it --rm -p 8888:8888 --name abhibook -v D:\ML\Docker\docker_first_notebook:/home/abhi/dX jupyter/scipy-notebook`
		* it’s launching a Jupyter Notebook container with some extra options.
		* `--name abhibook` → Assigns the container a name (notebook)
		* `-v D:\ML\Docker\docker_first_notebook:/home/abhi/dX` → Volume mount.
			- Maps your local folder `D:\ML\Docker\docker_first_notebook` → container folder `/home/abhi/dX`.
			- Any notebooks you `create/modify` inside Jupyter will be saved to your host machine.
		* jupyter/scipy-notebook → The Docker image. prebuilt Jupyter Notebook image from the Jupyter Docker Stacks
	* You open your browser at: `http://localhost:8888`
	* From this command `-v D:\ML\Docker\docker_first_notebook:/home/abhi/dX jupyter/scipy-notebook` you can not map you local directory with `/home/jovyan/work`. So in this casec you can not access this directory `/home/abhi/dX` from Left side bar.

	![can not access this directory](image-10.png)
	* To mount your local firectory with work run `docker run -d -it --rm -p 8888:8888 --name abhibook -v D:\ML\Docker\docker_first_notebook:/home/jovyan/work jupyter/scipy-notebook`
	* Why “jovyan” exists
		- The Jupyter Docker Stacks (like `jupyter/scipy-notebook`) are built with a non‑root user for security. That user is called `jovyan` (short for `Jupyter + Voyan` — a playful name chosen by the Jupyter team).
	* after this command

	![moun with work](image-11.png)

Any notebooks you create/modify inside Jupyter will be saved to your host machine.



## Install Python packages inside that container environment

- **Method 1**: docker exec -it <container id or name spae> sh
    - docker exec -it abhibook sh
    - Check Python and pip availability
        - `python --version` or `python3 --version`
        - `pip --version` or `pip3 --version`
        - Install the missing package `pip install <package-name>`
    - Since I used `--rm`, the container is deleted when stopped.
        - To keep installed packages, remove `--rm` from your run command.
        - `docker run -d -it --rm -p 8888:8888 --name abhibook -v D:\ML\Docker\docker_first_notebook:/home/jovyan/work jupyter/scipy-notebook` I am using this command. So install package I need to remove --rm ? If I do not remove --rm, in this situations when I recreate my container, I again I need install package ?
            - `--rm` means: remove the container automatically when it stops. Run without `--rm`
            - docker run -d -it -p 8888:8888 --name abhibook -v D:\ML\Docker\docker_first_notebook:/home/jovyan/work jupyter/scipy-notebook
- **Method 2**:Use shell incide 

![Use shell incide ](image-12.png)

- **Method 3** Build a custom image (recommended for reproducibility):
    - Install package, Create Docker file method.
		- Check Python and pip availability
			- Run `python --version` or `python3 --version`
			- Run `pip --version` or `pip3 --version`
		- Methos 1: create text file for all package and inastall into Docker cointasiner
			- `cd /d/ML/Docker/docker_first_notebook/Irish`
			- Create seprate package fror all python package
            	```command
				vi np_pkg.txt
				vi statemodel_pkg.txt
				vi viz_pkg.txt
				vi ml_pkg.txt
            	```
			- Create a new file named Dockerfile (no extension) in a folder on your machine
				- `vi Dockerfile`
				- Write the Dockerfile contents inside it
					```
					FROM jupyter/scipy-notebook	
					COPY np_pkg.txt statemodel_pkg.txt viz_pkg.txt ml_pkg.txt ./					
					RUN pip install -r np_pkg.txt \
						-r statemodel_pkg.txt \
						-r viz_pkg.txt \
						-r ml_pkg.txt
					```
                - Docker command TO BUILD PACKEG
					- `docker build -t mlx_pkg .`
				- See build process has been recorded & see Docker Desktop’s dashboard
					- `docker-desktop://dashboard/build/desktop-linux/desktop-linux/a1r9ksei4ghv2bo835ylovnyz`
					- Link can be changed
		- Method 2: Do not create separate .txt file. Just use only Docker file
			- `vi Dockerfile`
			- Write all packeg inside it
				```script
				FROM jupyter/scipy-notebook
				RUN pip install plotly sympy arviz scikit-learn pandas matplotlib seaborn
				```
			- Docker command TO BUILD PACKEG `docker build -t mlx_pkg .`
			- If you want a clean slate, delete the old image `docker rmi mlx_pkg`
- Finally Run Mount directory with Docker package
	- `docker run -d -it --rm -p 8888:8888 --name abhibook -v D:\ML\Docker\docker_first_notebook:/home/jovyan/work mlx_pkg`
	- **Note `mlx_pkg` added here and `jupyter/scipy-notebook` is removed.
	- GItbash terminal from windows I was getiing err.

## MLFLow + Notebook
#### Create a shared network
```
docker network inspect mlflow-net
docker network create mlflow-net
docker run -d --name mlflow-server --network mlflow-net -p 5000:5000 ghcr.io/mlflow/mlflow:v3.16.0 mlflow server --host 0.0.0.0 --port 5000
docker run -d -it --rm --name mlx --network mlflow-net -p 8888:8888 -v D:\ML\Docker\docker_first_notebook:/home/jovyan/work mlxpkg
mlflow.set_tracking_uri("http://172.19.0.2:5000")
```
#### Access from google crome

`http://localhost:5000`




https://docs.docker.com/reference/compose-file/build/ 

I need to understandf